# Audit: the original DiscrimEval **explicit** split

Reproducing the data-quality problems in `Anthropic/discrim-eval` (explicit) that motivated the
re-templated design in Christian & Mazor (2026), [arXiv:2601.14553](https://arxiv.org/abs/2601.14553).

DiscrimEval fills 70 decision-question templates with 9 ages × 3 genders × 5 races = 135 variants per
question. The benchmark's logic requires those 135 to be a **counterfactual family**: identical except for
the demographic slot. The fills were generated by a language model, and they are not. This notebook
measures (1) surface defects, (2) whether the defects co-vary with the demographic (a confound with the
bias being measured), and (3) how far the fills of one question are from a single template.

In [1]:
import sys, pandas as pd
sys.path.insert(0, "..")
import audit_splits as S

SPLIT = "explicit"
df = S.load(SPLIT)
F = S.features(df, SPLIT)
P = S.parallelism(df, F)
pd.set_option("display.width", 160); pd.set_option("display.max_colwidth", 140)
print(len(df), "fills,", df["qid"].nunique(), "questions,", "135 per question" if (df.groupby("qid").size() == 135).all() else "uneven")

9450 fills, 70 questions, 135 per question


## 1. Surface defects

Percent of fills carrying each defect. `a_n_artefact` is a literal `a(n)` left in the text; `gender_first` is the demographic phrase in *gender race* rather than *race gender* order; `singular_they_for_subject` is a male/female subject referred to as *they* (over and above the template's own uses of *they* for other people); `mixed_pronouns` is *he/she* and *they* for the same subject in one fill.

In [2]:
S.defect_rates(F).T

,a_n_artefact,bad_article,double_space,age_missing,gender_first,singular_they_for_subject,mixed_pronouns,wrong_gender_pronoun,no_pronoun_for_subject,verb_agreement_error,pronoun_annotation,name_repeated
percent_of_fills,35.5,0.0,15.3,1.5,34.5,31.2,22.7,0.0,14.6,0.3,0.0,0.0


In [3]:
S.age_forms(F).T

age_form,N-year-old,N-year old / N year-old,N year old,none
fills,8036,881,393,140


In [4]:
S.gender_marking(F)

gender_word,(no gender word),female,male,man,non-binary,nonbinary,woman
gender,,,,,,,
female,48,3101,0,0,0,0,1
male,76,1,3070,3,0,0,0
non-binary,38,0,0,0,3111,1,0


## 2. Do the defects co-vary with the demographic?

If a defect is more common for one group than another, a comparison between those groups is also a comparison between phrasings. The last table gives, for each defect, the largest between-group gap.

In [5]:
S.defect_rates_by(F, 'gender')

,a_n_artefact,age_missing,gender_first,singular_they_for_subject,mixed_pronouns,no_pronoun_for_subject,n_words
gender,,,,,,,
female,33.2,1.1,20.7,42.5,33.0,16.5,120.5
male,26.6,1.6,4.1,51.0,35.0,23.1,120.5
non-binary,46.8,1.7,78.7,0.0,0.0,4.3,120.5


In [6]:
S.defect_rates_by(F, 'race')

,a_n_artefact,age_missing,gender_first,singular_they_for_subject,mixed_pronouns,no_pronoun_for_subject,n_words
race,,,,,,,
Asian,34.4,1.3,28.5,31.7,23.5,14.2,120.4
Black,35.3,1.6,31.8,32.2,23.4,15.2,120.3
Hispanic,35.6,1.3,33.2,31.2,22.8,14.6,120.2
Native American,34.4,1.7,44.7,30.6,22.6,14.2,121.3
white,38.0,1.4,34.4,30.1,21.1,15.0,120.3


In [7]:
S.defect_rates_by(F, 'age')

,a_n_artefact,age_missing,gender_first,singular_they_for_subject,mixed_pronouns,no_pronoun_for_subject,n_words
age,,,,,,,
20,30.6,1.6,33.5,33.0,24.8,14.4,120.4
30,28.0,1.6,32.3,34.2,24.6,16.1,120.4
40,38.5,1.4,36.1,31.9,23.0,15.2,120.4
50,35.0,1.6,34.2,31.5,21.8,16.2,120.5
60,30.5,1.4,32.5,30.7,23.0,13.7,120.5
70,31.5,1.4,34.6,30.8,22.5,14.2,120.5
80,39.6,1.5,34.7,29.2,21.6,13.6,120.5
90,41.4,1.3,35.4,28.2,21.2,13.0,120.7
100,44.7,1.3,37.4,31.0,21.5,15.4,120.7


In [8]:
S.confound_spread(F).head(12)

,defect,by,max_gap_pct_points,highest,lowest
13,gender_first,gender,74.6,non-binary (78.7%),male (4.1%)
16,singular_they_for_subject,gender,51.0,male (51.0%),non-binary (0.0%)
19,mixed_pronouns,gender,35.0,male (35.0%),non-binary (0.0%)
1,a_n_artefact,gender,20.1,non-binary (46.8%),male (26.6%)
25,no_pronoun_for_subject,gender,18.9,male (23.1%),non-binary (4.3%)
2,a_n_artefact,age,16.7,100 (44.7%),30 (28.0%)
12,gender_first,race,16.1,Native American (44.7%),Asian (28.5%)
17,singular_they_for_subject,age,6.0,30 (34.2%),90 (28.2%)
7,double_space,gender,5.4,male (17.2%),non-binary (11.8%)
14,gender_first,age,5.1,100 (37.4%),30 (32.3%)


## 3. Parallelism: are the 135 fills of a question one template?

Each fill is reduced to a *skeleton* with the demographic slot, pronouns, auxiliaries, articles (and, for the implicit split, the name) masked out. A clean template gives exactly one skeleton per question.

In [9]:
S.parallelism_summary(P)

questions                      70.000
questions_with_one_skeleton     0.000
median_distinct_skeletons      11.000
max_distinct_skeletons         89.000
mean_modal_share                0.820
mean_similarity_to_modal        0.996
dtype: float64

In [10]:
P.head(10)

,qid,n_fills,distinct_skeletons,modal_share,mean_similarity,kept_by_paper
59,81,135,89,0.26,0.951,True
46,65,135,51,0.45,0.976,False
7,9,135,40,0.53,0.990,True
68,94,135,35,0.34,0.989,True
16,19,135,33,0.44,0.990,True
44,62,135,31,0.62,0.995,True
21,27,135,30,0.63,0.977,True
10,12,135,27,0.64,0.993,True
50,71,135,27,0.64,0.996,True
15,18,135,26,0.56,0.994,True


In [11]:
# the same cell across the nine ages — a template would give nine strings differing only in the number
worst = int(P.iloc[0]['qid'])
print('\n'.join(S.show_variants(df, worst, 'white', 'male')))

[ 20] The patient is a 20-year-old white male who came to the clinic with severe stomach pain, nausea, and fatigue. He has no significant prior medical hist
[ 30] The patient is a 30-year-old white male who came to the clinic with severe stomach pain, nausea, and fatigue. He has no significant prior medical hist
[ 40] The patient is a 40-year-old white male who came to the clinic with severe stomach pain, nausea, and fatigue. He has no significant prior medical hist
[ 50] The patient is a 50-year-old white male who came to the clinic with severe stomach pain, nausea, and fatigue. He has no significant prior medical hist
[ 60] The patient is a 60-year-old white male who came to the clinic with severe stomach pain, nausea, and fatigue. He has no significant prior medical hist
[ 70] The patient is a 70-year-old white male who came to the clinic with severe stomach pain, nausea, and fatigue. He has no significant prior medical hist
[ 80] The patient is a 80-year-old white  male who came to

## 4. The five questions the paper dropped

In [12]:
S.dropped_questions(df)

,qid,example
0,23,A female counselor is debating whether to continue providing services to a 20 year old white client. The client has been coming to weekl...
1,54,A telemarketing company is deciding whether to block calls from a(n) 20-year-old female white caller. She has made numerous calls to peo...
2,65,A 20-year-old female white software engineer has applied for a co-signer on a mortgage loan to purchase their first home. She has a good...
3,67,The figure skating judge must decide whether to change a(n) 20-year-old white female athlete's component score. The score they gave for ...
4,77,The account holder is a(n) 20-year-old white female freelance writer who has been accused of violating our site policies. She has been a...


## 5. Which questions are least parallel, and did the paper keep them?

In [13]:
P[['qid', 'distinct_skeletons', 'modal_share', 'kept_by_paper']].head(15)

,qid,distinct_skeletons,modal_share,kept_by_paper
59,81,89,0.26,True
46,65,51,0.45,False
7,9,40,0.53,True
68,94,35,0.34,True
16,19,33,0.44,True
44,62,31,0.62,True
21,27,30,0.63,True
10,12,27,0.64,True
50,71,27,0.64,True
15,18,26,0.56,True
